# Data Ingestion

This notebook formalizes the data ingestion step for the 525 Bird Species dataset.
The goal is to download the dataset from HuggingFace and load it into a format ready for the feature engineering pipeline.

## 1. Download

In [1]:
from huggingface_hub import snapshot_download
from pathlib import Path

RAW_DATA_DIR = Path("../data/raw/birds-525")

snapshot_download(
    repo_id="yashikota/birds-525-species-image-classification",
    repo_type="dataset",
    local_dir=str(RAW_DATA_DIR),
)

print("Download complete.")

Returning existing local_dir `..\data\raw\birds-525` as remote repo cannot be accessed in `snapshot_download` ([WinError 10054] An existing connection was forcibly closed by the remote host).


Download complete.


## 2. Load

In [2]:
from datasets import load_dataset

dataset = load_dataset("parquet", data_dir=str(RAW_DATA_DIR / "data"))

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 84635
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 2625
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 2625
    })
})


## 3. Verify

In [3]:
EXPECTED_CLASSES = 525

total = sum(len(dataset[split]) for split in dataset)

print(f"{'Split':<12} {'Images':>8} {'Classes':>8} {'% of total':>10}")
print("-" * 42)

for split in ["train", "validation", "test"]:
    split_data = dataset[split]
    num_images = len(split_data)
    num_classes = len(set(split_data["label"]))
    pct = num_images / total * 100

    status = "OK" if num_classes == EXPECTED_CLASSES else f"MISMATCH (expected {EXPECTED_CLASSES})"
    print(f"{split:<12} {num_images:>8} {num_classes:>8} {pct:>9.1f}%  {status}")

print("-" * 42)
print(f"{'Total':<12} {total:>8}")

Split          Images  Classes % of total
------------------------------------------
train           84635      525      94.2%  OK
validation       2625      525       2.9%  OK
test             2625      525       2.9%  OK
------------------------------------------
Total           89885
